# The Efficient Frontier (Section 2.1)

Following Markowitz (1952), the investor solves the constrained problem (2.1):

$$\min_{\boldsymbol{w}} \ \tfrac{1}{2}\,\boldsymbol{w}^{\top}\Sigma\,\boldsymbol{w}
\quad \text{subject to} \quad \boldsymbol{1}^{\top}\boldsymbol{w}=1,\quad \boldsymbol{\mu}^{\top}\boldsymbol{w}=R_{\min}.$$

Writing $A = \boldsymbol{1}^{\top}\Sigma^{-1}\boldsymbol{1}$, $B = \boldsymbol{1}^{\top}\Sigma^{-1}\boldsymbol{\mu}$, $C = \boldsymbol{\mu}^{\top}\Sigma^{-1}\boldsymbol{\mu}$, the optimal weights are the affine two-fund solution (2.2):

$$\boldsymbol{w}^{\star}(R_{\min})
= \frac{C - R_{\min}B}{AC - B^{2}}\,\Sigma^{-1}\boldsymbol{1}
+ \frac{R_{\min}A - B}{AC - B^{2}}\,\Sigma^{-1}\boldsymbol{\mu}.$$

As $R_{\min}$ varies, $\boldsymbol{w}^{\star}(R_{\min})$ traces out the frontier, whose variance is

$$\sigma^{2}(R_{\min}) = \frac{A\,R_{\min}^{2} - 2B\,R_{\min} + C}{AC - B^{2}},$$

minimised at the **global minimum-variance portfolio**: $R_{\mathrm{gmv}} = B/A$ with variance $\sigma^{2}_{\mathrm{gmv}} = 1/A$ (volatility $1/\sqrt{A}$), and $\boldsymbol{w}_{\mathrm{gmv}} = \Sigma^{-1}\boldsymbol{1}/A$.

The figure below is **purely illustrative**: the inputs $(\boldsymbol{\mu}, \Sigma)$ are arbitrary and the axes carry no numerical scale — the point is the shape of the frontier and the fact that changing $R_{\min}$ moves the optimal portfolio along it.

### Adding a risk-free asset

Markowitz (1952) has no risk-free asset. Following Tobin (1958), lending or borrowing at
a rate $r_f$ adds a straight line from $(0, r_f)$ through any risky portfolio, and the
best such line is the one tangent to the frontier. Its point of tangency is the
**tangency portfolio**

$$\boldsymbol{w}_{\mathrm{tan}}(r_f)
= \frac{\Sigma^{-1}(\boldsymbol{\mu} - r_f\boldsymbol{1})}
       {\boldsymbol{1}^{\top}\Sigma^{-1}(\boldsymbol{\mu} - r_f\boldsymbol{1})},$$

with

$$R_T = \frac{C - Br_f}{B - Ar_f}, \qquad
\sigma_T = \frac{\sqrt{C - 2r_fB + r_f^2A}}{B - Ar_f},$$

and the line itself — the **capital market line** — has slope
$\sqrt{C - 2r_fB + r_f^2A}$, which is exactly the Sharpe ratio of $T$.

Note that $\boldsymbol{w}_{\mathrm{tan}}$ **depends on $r_f$**: raising the risk-free rate
slides the tangency point up and to the right along the frontier. The commonly quoted
form $\Sigma^{-1}\boldsymbol{\mu}/(\boldsymbol{1}^{\top}\Sigma^{-1}\boldsymbol{\mu})$ is
the special case $r_f = 0$, or equivalently the same expression with $\boldsymbol{\mu}$
read as a vector of *excess* returns. The figure uses $r_f < B/A$, which is what puts $T$
on the efficient branch.

In [5]:
import numpy as np
import matplotlib.pyplot as plt

# Arbitrary illustrative inputs (any mu, positive-definite Sigma gives the same picture)
mu = np.array([0.06, 0.08, 0.10, 0.12, 0.15])
vols = np.array([0.12, 0.15, 0.18, 0.22, 0.28])
corr = 0.10 * np.ones((5, 5)) + 0.90 * np.eye(5)   # low correlation -> vertex close to the axis, sharp nose
Sigma = np.outer(vols, vols) * corr

ones = np.ones(len(mu))
Sigma_inv = np.linalg.inv(Sigma)

A = ones @ Sigma_inv @ ones
B = ones @ Sigma_inv @ mu
C = mu @ Sigma_inv @ mu
Delta = A * C - B**2

def w_star(R_min):
    # Two-fund solution (2.2)
    return ((C - R_min * B) / Delta) * (Sigma_inv @ ones) \
         + ((R_min * A - B) / Delta) * (Sigma_inv @ mu)

def frontier_sigma(R_min):
    return np.sqrt((A * R_min**2 - 2 * B * R_min + C) / Delta)

R_gmv, sigma_gmv = B / A, 1 / np.sqrt(A)   # variance at the GMV is 1/A

# --- the risk-free asset and the tangency portfolio (Tobin 1958) ---
# w_tan is proportional to Sigma^{-1}(mu - rf*1), NOT to Sigma^{-1}mu: the tangency
# portfolio moves as rf moves. The rf-free form Sigma^{-1}mu/(1'Sigma^{-1}mu) is the
# special case rf = 0 (equivalently, mu read as a vector of EXCESS returns).
rf = 0.055                                  # any rf < B/A puts T on the efficient branch
assert rf < R_gmv, "rf must sit below the GMV return for a sensible tangency"

def tangency(rf):
    w = Sigma_inv @ (mu - rf * ones)
    return w / w.sum()

den   = B - A * rf
R_T   = (C - B * rf) / den                          # mu' w_tan
sig_T = np.sqrt(C - 2 * rf * B + rf**2 * A) / den   # sqrt(w_tan' Sigma w_tan)
cml_slope = np.sqrt(C - 2 * rf * B + rf**2 * A)     # Sharpe ratio of T = slope of the CML

# consistency check against the explicit weights
w_T = tangency(rf)
assert np.isclose(mu @ w_T, R_T) and np.isclose(np.sqrt(w_T @ Sigma @ w_T), sig_T)

In [6]:
R_grid = np.linspace(R_gmv - 0.135, R_gmv + 0.20, 900)  # long branches -> sharp nose;
                                                        # the dominated branch is trimmed
sig_grid = frontier_sigma(R_grid)
efficient = R_grid >= R_gmv
_ylo = R_grid.min() * 1.10        # bottom of the axis, used by the guides and edge labels

fig, ax = plt.subplots(figsize=(9.6, 6.2))
ax.plot(sig_grid[efficient], R_grid[efficient], color="tab:blue", lw=2,
        label="Efficient frontier")
ax.plot(sig_grid[~efficient], R_grid[~efficient], color="tab:blue", lw=2,
        ls="--", alpha=0.5, label="Inefficient branch")

# --- the capital market line, tangent to the frontier at T ---
x_cml = np.array([0.0, sig_grid.max() * 1.55])
ax.plot(x_cml, rf + cml_slope * x_cml, color="tab:green", lw=1.8,
        label="Capital market line (CML)")
ax.scatter([0], [rf], color="tab:green", zorder=7, s=45, marker="s")
ax.scatter([sig_T], [R_T], color="tab:brown", zorder=8, s=70)
ax.annotate("Tangency portfolio $T$", (sig_T, R_T), xytext=(14, -16),
            textcoords="offset points", color="tab:brown", fontsize=10, fontweight="bold")
ax.plot([0, sig_T], [R_T, R_T], color="tab:brown", ls=":", lw=1)
ax.plot([sig_T, sig_T], [_ylo, R_T], color="tab:brown", ls=":", lw=1)
ax.text(-0.006, rf, r"$r_f$", ha="right", va="center", color="tab:green")
ax.text(sig_T, _ylo, r"$\sigma_T$", ha="center", va="top", color="tab:brown")
for x, lab, dy in [(0.40 * sigma_gmv, "lending", -15), (2.20 * sig_T, "borrowing", 9)]:
    ax.annotate(lab, xy=(x, rf + cml_slope * x), xytext=(0, dy), ha="center",
                textcoords="offset points", color="tab:green", fontsize=9)

# Global minimum-variance portfolio, with dotted guides to the axes
ax.scatter(sigma_gmv, R_gmv, color="tab:red", zorder=6)
ax.annotate("GMV", (sigma_gmv, R_gmv), xytext=(-10, -14), textcoords="offset points",
            color="tab:red", fontsize=10, fontweight="bold", ha="right")
ax.plot([0, sigma_gmv], [R_gmv, R_gmv], color="tab:red", ls=":", lw=1)
ax.plot([sigma_gmv, sigma_gmv], [_ylo, R_gmv], color="tab:red", ls=":", lw=1)
ax.text(sigma_gmv, _ylo, r"$1/\sqrt{A}$", ha="center", va="top", color="tab:red")
ax.text(-0.005, R_gmv, r"$B/A$", ha="right", va="center", color="tab:red")

# A target below B/A lands on the dominated branch: the GMV itself
# offers strictly higher return at strictly lower volatility
R_bad = R_gmv - 0.035
ax.scatter(frontier_sigma(R_bad), R_bad, color="tab:gray", zorder=5,
           label=r"$\boldsymbol{w}^{\star}(R_{\min})$ with $R_{\min} < B/A$ (dominated)")
ax.annotate("", xy=(sigma_gmv, R_gmv), xytext=(frontier_sigma(R_bad), R_bad),
            arrowprops=dict(arrowstyle="->", color="tab:gray", lw=1.2, ls=":",
                            shrinkA=6, shrinkB=6))
ax.text(frontier_sigma(R_bad) - 0.022, R_bad - 0.010,
        "GMV attains higher return\nat lower volatility",
        color="tab:gray", fontsize=9, va="top", ha="right")

# Individual assets: every single-name portfolio is feasible but not efficient, so it
# lies strictly to the RIGHT of the frontier. Scattered at random, keeping clear of the
# tip of the bullet where the labelled portfolios sit.
_rng = np.random.default_rng(7)
_Ra = np.concatenate([_rng.uniform(R_gmv + 0.030, R_gmv + 0.135, 5),
                      _rng.uniform(R_gmv - 0.100, R_gmv - 0.025, 3)])
_lo = frontier_sigma(_Ra) * 1.35                      # keep well clear of the frontier
_hi = np.maximum(_lo * 1.10, 0.90 * sig_grid.max() * 1.05)
_sa = _rng.uniform(_lo, _hi)
ax.scatter(_sa, _Ra, s=30, color="tab:purple", alpha=0.75, marker="^", zorder=4,
           label="individual assets")

# Schematic axes: no numerical scale
ax.set_xticks([]); ax.set_yticks([])
ax.set_xlim(0, sig_grid.max() * 1.05)
ax.set_ylim(_ylo, R_grid.max() * 1.06)
ax.set_xlabel(r"Volatility $\sigma$", labelpad=20)
ax.set_ylabel(r"Expected return $R_{\min}$", labelpad=28)
ax.set_title("Efficient frontier, CML, GMV and the tangency portfolio")
ax.legend(frameon=False, loc="lower right", fontsize=8.0, ncol=2, columnspacing=1.2)
fig.tight_layout()
fig.savefig("efficient_frontier.png", dpi=200, bbox_inches="tight")
plt.show()

**Reading the figure.** Each orange point is the optimal portfolio $\boldsymbol{w}^{\star}(R_{\min})$ for one value of $R_{\min}$; raising the target return moves the portfolio up along the efficient branch, at the cost of higher volatility. Only targets $R_{\min} > B/A$ are efficient. A target below $B/A$ (grey point) still solves (2.1), but lands on the dominated branch: the global minimum-variance portfolio at $(1/\sqrt{A},\, B/A)$ delivers a strictly higher return at strictly lower volatility, so no rational mean--variance investor would choose it.

With the risk-free asset in place, every efficient portfolio is a mix of $r_f$ and the
single risky portfolio $T$ — Tobin's separation theorem. Points on the capital market
line to the left of $T$ lend at $r_f$; points to the right borrow. The line lies weakly
above the frontier everywhere and touches it only at $T$, so once borrowing and lending
are available no portfolio on the risky frontier other than $T$ is held.